In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)


device: mps


In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

if device == "mps":
    clf = pipeline(
        "zero-shot-classification",
        model=model_name,
        tokenizer=model_name,
        framework="pt",
        device=torch.device("mps"),
    )
else:
    clf = pipeline(
        "zero-shot-classification",
        model=model_name,
        tokenizer=model_name,
        framework="pt",
        device=-1,
    )

candidate_labels = ["paraphrase", "contradiction", "unrelated"]
hypothesis_template = "The relationship between the two sentences is {}."

print(model_name)
print(candidate_labels)
print(hypothesis_template)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
['paraphrase', 'contradiction', 'unrelated']
The relationship between the two sentences is {}.


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
texts = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))
print(texts[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [4]:
batch_size = 16
preds = []
all_ranked_labels = []
all_ranked_scores = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i + batch_size]
    outputs = clf(
        batch_texts,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
        batch_size=batch_size,
        truncation=True,
    )

    if isinstance(outputs, dict):
        outputs = [outputs]

    for out in outputs:
        labels = list(out["labels"])
        scores = [float(s) for s in out["scores"]]
        all_ranked_labels.append(labels)
        all_ranked_scores.append(scores)
        preds.append(1 if labels[0] == "paraphrase" else 0)

y_pred = np.array(preds)
print("done")


  0%|          | 0/26 [00:00<?, ?it/s]

done


In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6764705882352942, 'f1': 0.8006042296072508}
                precision    recall  f1-score   support

not_paraphrase       0.44      0.09      0.14       129
    paraphrase       0.69      0.95      0.80       279

      accuracy                           0.68       408
     macro avg       0.57      0.52      0.47       408
  weighted avg       0.61      0.68      0.59       408



In [6]:
for i in range(5):
    score_map = {label: float(score) for label, score in zip(all_ranked_labels[i], all_ranked_scores[i])}
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "top_relation:", all_ranked_labels[i][0])
    print("scores:", {label: score_map[label] for label in candidate_labels})


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 top_relation: contradiction
scores: {'paraphrase': 0.004584910348057747, 'contradiction': 0.9948615431785583, 'unrelated': 0.0005535607342608273}
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1 top_relation: paraphrase
scores: {'paraphrase': 0.9487652778625488, 'contradiction': 0.03598674014210701, 'unrelated': 0.015247955918312073}
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against 

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    score_map = {label: float(score) for label, score in zip(all_ranked_labels[i], all_ranked_scores[i])}
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "top_relation:", all_ranked_labels[i][0])
    print("scores:", {label: score_map[label] for label in candidate_labels})


num_errors: 132
idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 top_relation: contradiction
scores: {'paraphrase': 0.004584910348057747, 'contradiction': 0.9948615431785583, 'unrelated': 0.0005535607342608273}
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1 top_relation: paraphrase
scores: {'paraphrase': 0.9487652778625488, 'contradiction': 0.03598674014210701, 'unrelated': 0.015247955918312073}
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on

In [8]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": device,
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6764705882352942,
 'f1': 0.8006042296072508}